# 01 — Data Exploration and Dataset Understanding
**Project:** Feature Extraction and Price Prediction for Mobile Phones (NextHikes — Project 4)
**Day 1 of 12 · Phase 1: Data Understanding**

### Objectives
1. Load the raw dataset without modifying it.
2. Verify its structure: shape, columns, data types.
3. Identify the target, the identifier, and the numerical / categorical / text-specification features.
4. Measure missing values (explicit *and* disguised), duplicates, and cardinality.
5. Investigate suspicious values and record evidence for each.
6. Produce a data dictionary and an initial data-quality report.

### Ground rules for this notebook
- **Read-only.** Nothing here cleans, drops, or overwrites data. Where a numeric view of a text column is needed
  (e.g. price), it is built as a *temporary* Series for inspection only. Actual cleaning is Day 3.
- **Evidence before decisions.** Every suspicious value is shown alongside the rows that explain it.
- **No model training.**

## 1. Environment and imports
`src/data_loader.py` holds the loading logic so that every later notebook and the final pipeline read the data in
exactly the same way. The notebook lives in `notebooks/`, so the project root is added to `sys.path`.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_data, RAW_DATA_PATH, EXPECTED_COLUMNS

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
RANDOM_STATE = 42

print("pandas :", pd.__version__)
print("numpy  :", np.__version__)
print("Data   :", RAW_DATA_PATH.relative_to(PROJECT_ROOT))

pandas : 3.0.5
numpy  : 2.5.3
Data   : data/raw/Processed_Flipdata.csv


## 2. Load the raw data and check dimensions

In [2]:
raw_df = load_raw_data()          # validates the expected 12-column schema
df = raw_df.copy()                # all inspection happens on a copy

print(f"Rows: {df.shape[0]}   Columns: {df.shape[1]}")
print("Columns:", list(df.columns))

Rows: 541   Columns: 12
Columns: ['Unnamed: 0', 'Model', 'Colour', 'Memory', 'RAM', 'Battery_', 'Rear Camera', 'Front Camera', 'AI Lens', 'Mobile Height', 'Processor_', 'Prize']


**Interpretation.** The file loads and matches the brief: **541 rows × 12 columns**, with the column names exactly as
listed. Two names carry trailing underscores (`Battery_`, `Processor_`) and the target is spelled **`Prize`** — these
are renamed on Day 3, not now, so that this notebook reflects the raw file.

## 3. First look at the records — `head()`, `tail()`, `sample()`
`head`/`tail` show the file's ordering; `sample` with a fixed seed gives a reproducible look at the middle.

In [3]:
display(df.head())
display(df.tail())
df.sample(5, random_state=RANDOM_STATE)

,Unnamed: 0,Model,Colour,Memory,RAM,Battery_,Rear Camera,Front Camera,AI Lens,Mobile Height,Processor_,Prize
0,0,Infinix SMART 7,Night Black,64,4,6000,13MP,5MP,1,16.76,Unisoc Spreadtrum SC9863A1,"7,299"
1,1,Infinix SMART 7,Azure Blue,64,4,6000,13MP,5MP,1,16.76,Unisoc Spreadtrum SC9863A1,"7,299"
2,2,MOTOROLA G32,Mineral Gray,128,8,5000,50MP,16MP,0,16.64,Qualcomm Snapdragon 680,"11,999"
3,3,POCO C50,Royal Blue,32,2,5000,8MP,5MP,0,16.56,Mediatek Helio A22,"5,649"
4,4,Infinix HOT 30i,Marigold,128,8,5000,50MP,5MP,1,16.76,G37,"8,999"


,Unnamed: 0,Model,Colour,Memory,RAM,Battery_,Rear Camera,Front Camera,AI Lens,Mobile Height,Processor_,Prize
536,637,SAMSUNG Galaxy S23 5G,Cream,256,8,3900,50MP,12MP,0,15.49,Qualcomm Snapdragon 8 Gen 2,"79,999"
537,638,LAVA Z21,Cyan,32,2,3100,5MP,2MP,0,12.70,Octa Core,"5,998"
538,639,Tecno Spark 8T,Turquoise Cyan,64,4,5000,50MP,8MP,0,16.76,MediaTek Helio G35,"9,990"
539,641,SAMSUNG Galaxy A54 5G,Awesome Lime,128,8,5000,50MP,32MP,0,16.26,"Exynos 1380, Octa Core","38,999"
540,642,OPPO A77,Sky Blue,128,4,5000,50MP,8MP,0,16.66,Mediatek Helio G35,"15,999"


,Unnamed: 0,Model,Colour,Memory,RAM,Battery_,Rear Camera,Front Camera,AI Lens,Mobile Height,Processor_,Prize
229,229,Infinix HOT 20 Play,Luna Blue,64,4,6000,13MP,8MP,1,17.32,MediaTek G37,"8,499"
73,73,REDMI A1+,Light Blue,32,2,5000,8MP,5MP,0,16.56,Mediatek Helio A22,"6,299"
352,362,POCO X4 Pro 5G,Laser Black,128,8,5000,64MP,16MP,0,16.94,Qualcomm Snapdragon 695 5G,"19,499"
86,86,realme C55,Rainforest,64,4,5000,64MP,8MP,0,17.07,Helio G88,"10,999"
470,536,realme Narzo 50A Prime,Flash Black,128,4,5000,50MP,8MP,0,16.76,MediaTek Helio G88,"12,599"


**Interpretation.**
- Each row is a **product listing variant**: a model in one colour with one storage/RAM configuration
  (e.g. *Infinix SMART 7 · Night Black · 64 GB · 4 GB*).
- `Prize` holds values such as `"7,299"` — a **string with thousands separators**, so it is not numeric yet.
- `Rear Camera` / `Front Camera` are strings with an `MP` suffix (`"13MP"`).
- `Mobile Height` values around 16–17 match how Flipkart lists **display size in cm** (16.76 cm ≈ 6.6 in).
  This is a hypothesis to keep testing, not an assumption: the column name says *height*, the values behave like a
  screen diagonal.
- `Unnamed: 0` looks like a leftover row index from an earlier export.

## 4. Data types — `info()`

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 541 entries, 0 to 540
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Unnamed: 0     541 non-null    int64  
 1   Model          541 non-null    str    
 2   Colour         541 non-null    str    
 3   Memory         541 non-null    int64  
 4   RAM            541 non-null    int64  
 5   Battery_       541 non-null    int64  
 6   Rear Camera    541 non-null    str    
 7   Front Camera   541 non-null    str    
 8   AI Lens        541 non-null    int64  
 9   Mobile Height  541 non-null    float64
 10  Processor_     541 non-null    str    
 11  Prize          541 non-null    str    
dtypes: float64(1), int64(5), str(6)
memory usage: 50.8 KB


**Interpretation.** No column has nulls (541 non-null everywhere). Five columns are numeric
(`Memory`, `RAM`, `Battery_`, `AI Lens`, `Mobile Height`) plus the index. **`Prize`, `Rear Camera` and `Front Camera`
are stored as text even though they represent numbers** — they need parsing before any numerical analysis.

## 5. Descriptive statistics

In [5]:
numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
text_cols = [c for c in df.columns if c not in numeric_cols]

display(df[numeric_cols].describe().T)
df[text_cols].describe().T

,count,mean,std,min,25%,50%,75%,max
Unnamed: 0,541.0,289.711645,182.359185,0.0,135.00,273.00,434.00,642.00
Memory,541.0,110.550832,60.600694,16.0,64.00,128.00,128.00,256.00
RAM,541.0,5.397412,1.984923,2.0,4.00,6.00,8.00,8.00
Battery_,541.0,4871.587800,780.148862,800.0,5000.00,5000.00,5000.00,7000.00
AI Lens,541.0,0.062847,0.242911,0.0,0.00,0.00,0.00,1.00
Mobile Height,541.0,16.431201,2.523553,4.5,16.51,16.71,16.94,41.94


,count,unique,top,freq
Model,541,187,realme C55,9
Colour,541,275,Yellow,12
Rear Camera,541,14,50MP,260
Front Camera,541,13,16MP,164
Processor_,541,123,Qualcomm Snapdragon 680,33
Prize,541,158,"15,999",23


**Interpretation — numeric columns**
- **Memory** (16–256) and **RAM** (2–8) take only a handful of discrete values — they are *ordinal-like*, not continuous.
  RAM's maximum of **8** is worth noting: several premium phones in the market ship with 12 GB (checked in §11).
- **Battery_** median is 5000 with a min of **800** — values near 800–2550 cannot belong to modern smartphones (§11).
- **AI Lens** has mean 0.063 → only ~6 % of rows have the flag set.
- **Mobile Height** spans 4.5 to **41.94**; the 75th percentile is 16.94, so 41.94 is far outside the normal range (§11).

**Interpretation — text columns**
- `Model` has 187 distinct values over 541 rows, `Colour` 275, `Processor_` 123 → all **high-cardinality**.
- `Prize` has 158 distinct values; the most frequent is listed in the `top` column above.

## 6. Feature roles
Classifying every column before analysis avoids accidentally modelling an ID or leaking the target.

In [6]:
TARGET = "Prize"
column_roles = pd.DataFrame({
    "column": df.columns,
    "stored_dtype": [str(df[c].dtype) for c in df.columns],
    "role": [
        "identifier (row index, not a feature)",
        "categorical – high cardinality (product name)",
        "categorical – high cardinality",
        "numeric – discrete (storage, GB)",
        "numeric – discrete (RAM, GB)",
        "numeric – continuous (battery, mAh)",
        "text spec → numeric (primary rear camera, MP)",
        "text spec → numeric (front camera, MP)",
        "binary flag (0/1)",
        "numeric – continuous (display size, cm — to verify)",
        "categorical – high cardinality (chipset text)",
        "TARGET – text → numeric (price, INR)",
    ],
})
column_roles

,column,stored_dtype,role
0,Unnamed: 0,int64,"identifier (row index, not a feature)"
1,Model,str,categorical – high cardinality (product name)
2,Colour,str,categorical – high cardinality
3,Memory,int64,"numeric – discrete (storage, GB)"
4,RAM,int64,"numeric – discrete (RAM, GB)"
5,Battery_,int64,"numeric – continuous (battery, mAh)"
6,Rear Camera,str,"text spec → numeric (primary rear camera, MP)"
7,Front Camera,str,"text spec → numeric (front camera, MP)"
8,AI Lens,int64,binary flag (0/1)
9,Mobile Height,float64,"numeric – continuous (display size, cm — to ve..."


**Target:** `Prize` (price in Indian Rupees — the dataset comes from Flipkart India and values are in the ₹1k–₹81k range).
**Not a feature:** `Unnamed: 0`.
**Candidate features:** the remaining 10 columns, of which 3 need parsing (`Rear Camera`, `Front Camera`) or
heavy feature engineering (`Processor_`, `Model`) on Day 4.

## 7. Unique values and cardinality

In [7]:
cardinality = pd.DataFrame({
    "n_unique": df.nunique(),
    "pct_unique": (df.nunique() / len(df) * 100).round(1),
}).sort_values("n_unique", ascending=False)
display(cardinality)

for col in ["Memory", "RAM", "AI Lens", "Rear Camera", "Front Camera"]:
    print(f"\n{col}:")
    print(df[col].value_counts().sort_index().to_dict())

,n_unique,pct_unique
Unnamed: 0,541,100.0
Colour,275,50.8
Model,187,34.6
Prize,158,29.2
Processor_,123,22.7
Mobile Height,38,7.0
Battery_,37,6.8
Rear Camera,14,2.6
Front Camera,13,2.4
Memory,5,0.9



Memory:
{16: 6, 32: 52, 64: 169, 128: 259, 256: 55}

RAM:
{2: 37, 3: 40, 4: 186, 6: 121, 8: 157}

AI Lens:
{0: 507, 1: 34}

Rear Camera:
{'0MP': 10, '108MP': 31, '12MP': 10, '13MP': 63, '16MP': 1, '200MP': 8, '2MP': 6, '32MP': 2, '40MP': 2, '48MP': 31, '50MP': 260, '5MP': 3, '64MP': 62, '8MP': 52}

Front Camera:
{'0MP': 16, '10MP': 2, '12MP': 6, '13MP': 49, '16MP': 164, '20MP': 2, '2MP': 3, '32MP': 22, '44MP': 2, '50MP': 7, '5MP': 135, '60MP': 4, '8MP': 129}


**Interpretation.**
- `Unnamed: 0` is 100 % unique — consistent with an index.
- `Colour` (275 values, ~51 % unique) carries little general information: names like *“Nebula Blue”* are brand
  marketing labels. One-hot encoding it would create ~275 sparse columns from 541 rows — a clear overfitting risk.
- `Model` (187) and `Processor_` (123) are also too granular to one-hot encode directly; Day 4 will extract structure
  from them (brand from `Model`; chipset family/tier from `Processor_`).
- **Memory** is concentrated at 64 GB and 128 GB; **RAM** at 4/6/8 GB.
- **AI Lens = 1 for only 34 rows.**
- **Rear camera** is dominated by 50 MP (260 rows). **`0MP` appears 10× (rear) and 16× (front)** — this most likely
  means *no camera* (or not recorded) rather than a real 0-megapixel sensor (§8).

## 8. Missing values — explicit and disguised
`isna()` only finds real NaNs. Scraped data often hides missing information as placeholder text, so we also check for
blank strings, `0MP` cameras, and processor fields that contain something other than a chipset.

In [8]:
explicit_missing = df.isna().sum()
print("Explicit NaN counts:", explicit_missing.to_dict())
print("Total explicit NaNs:", int(explicit_missing.sum()))

blank_strings = {c: int(df[c].astype(str).str.strip().eq("").sum()) for c in text_cols}
print("\nBlank strings per text column:", blank_strings)

NON_CHIPSET = ["Android", "Oxygen OS", "ios", "IOS", "Octa Core", "Octa-core", "6 Core",
               "2GHz, Quad Core ", "MediaTek", "Unisoc", "Spreadtrum", "0 0 0 208MHz"]
disguised = pd.Series({
    "Rear Camera == '0MP'": (df["Rear Camera"] == "0MP").sum(),
    "Front Camera == '0MP'": (df["Front Camera"] == "0MP").sum(),
    "Processor_ is OS / core count / vendor only": df["Processor_"].isin(NON_CHIPSET).sum(),
    "Colour == 'M3)' (garbled)": (df["Colour"] == "M3)").sum(),
})
disguised

Explicit NaN counts: {'Unnamed: 0': 0, 'Model': 0, 'Colour': 0, 'Memory': 0, 'RAM': 0, 'Battery_': 0, 'Rear Camera': 0, 'Front Camera': 0, 'AI Lens': 0, 'Mobile Height': 0, 'Processor_': 0, 'Prize': 0}
Total explicit NaNs: 0

Blank strings per text column: {'Model': 0, 'Colour': 0, 'Rear Camera': 0, 'Front Camera': 0, 'Processor_': 0, 'Prize': 0}


Rear Camera == '0MP'                           10
Front Camera == '0MP'                          16
Processor_ is OS / core count / vendor only    25
Colour == 'M3)' (garbled)                       5
dtype: int64

In [9]:
df[df["Processor_"].isin(NON_CHIPSET)][["Model", "Processor_", "Prize"]]

,Model,Processor_,Prize
133,Micromax 3,MediaTek,"2,615"
152,micromax 3,0 0 0 208MHz,"2,490"
156,micromax 3,MediaTek,"2,499"
193,micromax 2,MediaTek,"2,499"
243,redmi max,Octa Core,"4,999"
246,redmi min,Spreadtrum,920
247,redmi min,Spreadtrum,920
330,I Kall Z19Pro Flash blue,Octa Core,"7,999"
338,I Kall Z19Pro,Octa Core,"7,999"
346,OPPO A78 5G,Android,"19,280"


**Interpretation.**
- **Zero explicit missing values** — but that does not mean the data is complete.
- **Disguised missingness exists:**
  - `0MP` cameras: on inspection (§11) these rows are basic feature phones; “0 MP” means *no camera*, which is a real,
    meaningful value for price — it should be kept as 0, not imputed.
  - **~25 rows have a `Processor_` that is not a chipset**: an operating system (`Oxygen OS`, `Android`, `ios`),
    a core count (`Octa Core`, `6 Core`), only a vendor (`MediaTek`, `Unisoc`), or noise (`0 0 0 208MHz`).
    For these, the specific chipset is effectively unknown. Day 4 must handle an *unknown* processor category.
  - `Colour == "M3)"` on 5 rows is a parsing fragment, not a colour.

## 9. Duplicates and near-duplicates
Three levels matter here:
1. **Exact duplicate rows** including the index → would indicate a copy-paste of the file.
2. **Duplicate rows ignoring `Unnamed: 0`** → the same listing scraped twice.
3. **Same model + configuration in different colours** → not errors, but *near-duplicates* that affect how we split
   train/test data.

In [10]:
features_only = df.drop(columns="Unnamed: 0")
print("Exact duplicate rows (incl. index)  :", df.duplicated().sum())
print("Duplicate rows ignoring the index   :", features_only.duplicated().sum())

dupes = df[features_only.duplicated(keep=False)].sort_values(["Model", "Colour", "Unnamed: 0"])
dupes

Exact duplicate rows (incl. index)  : 0
Duplicate rows ignoring the index   : 10


,Unnamed: 0,Model,Colour,Memory,RAM,Battery_,Rear Camera,Front Camera,AI Lens,Mobile Height,Processor_,Prize
46,46,MOTOROLA e32,Arctic Blue,64,4,5000,50MP,8MP,0,16.51,Mediatek Helio G37,"7,999"
48,48,MOTOROLA e32,Arctic Blue,64,4,5000,50MP,8MP,0,16.51,Mediatek Helio G37,"7,999"
479,553,POCO C31,Royal Blue,64,4,5000,13MP,5MP,0,16.59,MediaTek G35,"8,999"
480,556,POCO C31,Royal Blue,64,4,5000,13MP,5MP,0,16.59,MediaTek G35,"8,999"
350,360,POCO X4 Pro 5G,Laser Black,128,8,5000,64MP,16MP,0,16.94,Qualcomm Snapdragon 695 5G,"19,499"
352,362,POCO X4 Pro 5G,Laser Black,128,8,5000,64MP,16MP,0,16.94,Qualcomm Snapdragon 695 5G,"19,499"
90,90,SAMSUNG Galaxy F14 5G,B.A.E. Purple,128,4,6000,50MP,13MP,0,16.76,"Exynos 1330, Octa Core","13,490"
91,91,SAMSUNG Galaxy F14 5G,B.A.E. Purple,128,4,6000,50MP,13MP,0,16.76,"Exynos 1330, Octa Core","13,490"
130,130,SAMSUNG Galaxy F14 5G,OMG Black,128,6,6000,50MP,13MP,0,16.76,"Exynos 1330, Octa Core","14,490"
132,132,SAMSUNG Galaxy F14 5G,OMG Black,128,6,6000,50MP,13MP,0,16.76,"Exynos 1330, Octa Core","14,490"


**Interpretation.** No row is copied verbatim (the index differs), but **10 rows are repeats of another row on every
real field** — 20 rows forming 10 pairs. These are the same listing captured twice. They are candidates for removal on
Day 3, because a duplicated phone could land in both train and test sets and inflate test scores.

In [11]:
variant_keys = ["Model", "Colour", "Memory", "RAM"]
price_conflicts = (df.groupby(variant_keys)["Prize"].nunique().loc[lambda s: s > 1])
print("Identical variant (model, colour, memory, RAM) with >1 price:", len(price_conflicts))
df.set_index(variant_keys).loc[price_conflicts.index, ["Unnamed: 0", "Prize"]]

Identical variant (model, colour, memory, RAM) with >1 price: 4


Unnamed: 0   Prize
Model        Colour         Memory RAM                    
MOTOROLA g42 Atlantic Green 64     4           164  11,248
                                   4           291  13,999
OPPO A17     Lake Blue      64     4           183  12,499
                                   4           249  11,970
POCO C50     Royal Blue     32     2             3   5,649
                                   2            53   6,999
POCO F5 5G   Carbon Black   256    8           282  29,999
                                   8           456  32,999

**Interpretation.** Four variants appear with **two different prices** for the same specification (e.g. POCO C50
32 GB/2 GB at ₹5,649 and ₹6,999). These are most plausibly listings from different sellers or dates. They are not
errors; they show that **price has irreducible noise** even for identical specifications — no model can be perfect here.

In [12]:
model_norm = df["Model"].str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
rows_per_model = model_norm.value_counts()
config_count = df.assign(m=model_norm).groupby(["m", "Memory", "RAM"]).ngroups

print("Distinct models (raw)            :", df["Model"].nunique())
print("Distinct models (case-normalised):", model_norm.nunique())
print("Distinct model+memory+RAM configs:", config_count)
print("\nRows per model:")
print(rows_per_model.describe().round(2).to_dict())
print("\nModels with the most rows:")
print(rows_per_model.head(8).to_dict())

Distinct models (raw)            : 187
Distinct models (case-normalised): 181
Distinct model+memory+RAM configs: 280

Rows per model:
{'count': 181.0, 'mean': 2.99, 'std': 1.99, 'min': 1.0, '25%': 1.0, '50%': 2.0, '75%': 4.0, 'max': 9.0}

Models with the most rows:
{'realme c55': 9, 'redmi note 12 pro 5g': 9, 'vivo y16': 9, 'samsung galaxy a14 5g': 9, 'infinix hot 30i': 8, 'redmi 10': 8, 'vivo t2x 5g': 8, 'samsung galaxy f14 5g': 8}


**Interpretation — this is one of the most important Day 1 findings.**
541 rows describe only **~181 distinct phone models** and **~280 distinct hardware configurations**. A typical model
appears 2–4 times (max 9), usually differing only by colour.

If we use a plain random train/test split, the *same phone in a different colour* will sit in both sets. The model can
then “recognise” the phone rather than learn how specifications drive price, and test scores will be optimistic.
**Implication for Day 7:** evaluate with a **grouped split (`GroupShuffleSplit` / `GroupKFold`) by model**, so the
test set contains phones the model has never seen. We'll report random-split scores alongside for comparison.

## 10. The target: `Prize`
To describe the target we need numbers. We build a **temporary** numeric Series; `df` is not modified.

In [13]:
price_str = df["Prize"].astype(str)
is_valid_format = price_str.str.fullmatch(r"\d{1,3}(,\d{3})*|\d+")
print("Rows matching a clean number format:", int(is_valid_format.sum()), "/", len(df))

price_tmp = pd.to_numeric(price_str.str.replace(",", "", regex=False), errors="coerce")
print("Values that failed to convert      :", int(price_tmp.isna().sum()))

q1, q3 = price_tmp.quantile([0.25, 0.75])
iqr = q3 - q1
upper_fence = q3 + 1.5 * iqr
summary = pd.Series({
    "min": price_tmp.min(), "Q1": q1, "median": price_tmp.median(), "mean": round(price_tmp.mean(), 1),
    "Q3": q3, "max": price_tmp.max(), "std": round(price_tmp.std(), 1),
    "skewness": round(price_tmp.skew(), 3),
    "IQR upper fence": upper_fence, "rows above fence": int((price_tmp > upper_fence).sum()),
    "rows below 3000": int((price_tmp < 3000).sum()),
})
summary

Rows matching a clean number format: 541 / 541
Values that failed to convert      : 0


min                   920.000
Q1                   9290.000
median              13499.000
mean                16228.400
Q3                  19280.000
max                 80999.000
std                 10793.600
skewness                2.426
IQR upper fence     34265.000
rows above fence       25.000
rows below 3000        15.000
dtype: float64

In [14]:
above = df.loc[price_tmp > upper_fence, ["Model", "Memory", "RAM", "Prize"]]
above.assign(price=price_tmp[above.index]).sort_values("price", ascending=False).drop(columns="price")

,Model,Memory,RAM,Prize
493,APPLE iPhone 14 Plus,128,6,"80,999"
520,APPLE iPhone 14 Plus,128,3,"80,999"
525,APPLE iPhone 14 Plus,128,4,"80,999"
536,SAMSUNG Galaxy S23 5G,256,8,"79,999"
426,Google Pixel 7,128,8,"55,999"
417,Google Pixel 7,128,8,"55,999"
416,Google Pixel 7,128,8,"55,999"
532,APPLE iPhone 12,64,8,"53,999"
222,SAMSUNG Galaxy S22 Plus 5G,128,8,"49,999"
400,MOTOROLA Edge 30 Ultra,256,4,"49,999"


**Interpretation.**
- **Every price parses cleanly** after removing commas — no currency symbols or text. (Values under 1,000 such as `920`
  simply have no comma.)
- Prices range from **₹920 to ₹80,999**; median ₹13,499 vs mean ₹16,228 → **strong right skew (≈2.43)**.
  Most phones are budget/mid-range; a thin tail of premium devices pulls the mean up.
- **25 rows exceed the IQR upper fence (~₹34,265).** They are recognisable premium phones (iPhone 14 Plus, Galaxy S23,
  Pixel 7, Edge 30 Ultra). These are **genuine extreme observations, not errors** — removing them would make the model
  useless for premium pricing.
- **Implication for modelling:** consider predicting `log(price)` (Day 7–8) and favour MAE alongside RMSE, since RMSE
  is dominated by the expensive tail.

## 11. Investigating suspicious values
Each subsection states the suspicion, shows the evidence, and records what is (and is not) concluded.

### 11a. Feature-phone rows inherit values from the row above
16 rows have a battery under 2,600 mAh or a display under 10 cm — basic feature phones (Micromax, “redmi min/max/mono”).
Printing each one *with the row directly above it* reveals a pattern.

In [15]:
feature_phone_mask = (df["Battery_"] < 2600) | (df["Mobile Height"] < 10)
print("Feature-phone-like rows:", int(feature_phone_mask.sum()))

cols = ["Unnamed: 0", "Model", "Colour", "Memory", "RAM", "Battery_", "Rear Camera", "Mobile Height", "Prize"]
pairs = []
for i in df.index[feature_phone_mask]:
    if i == 0:
        continue
    prev, cur = df.loc[i - 1], df.loc[i]
    pairs.append({
        "row": i, "model": cur["Model"], "prev_model": prev["Model"],
        "colour": cur["Colour"], "colour_same_as_prev": cur["Colour"] == prev["Colour"],
        "mem_ram": f'{cur["Memory"]}/{cur["RAM"]}', "prev_mem_ram": f'{prev["Memory"]}/{prev["RAM"]}',
        "battery": cur["Battery_"], "price": cur["Prize"],
    })
carry_over = pd.DataFrame(pairs)
print("Colour identical to previous row:", int(carry_over["colour_same_as_prev"].sum()), "of", len(carry_over))
carry_over

Feature-phone-like rows: 16
Colour identical to previous row: 16 of 16


,row,model,prev_model,colour,colour_same_as_prev,mem_ram,prev_mem_ram,battery,price
0,98,Micromax IN 2C,realme 10 Pro 5G,Nebula Blue,True,128/6,128/6,800,"1,449"
1,108,micromax 2,realme Narzo 30 Pro 5G,Blade Silver,True,128/8,128/8,1900,"1,849"
2,116,micromax,SAMSUNG Galaxy F14 5G,OMG Black,True,128/4,128/4,1000,"1,422"
3,118,micromax 1,MOTOROLA Edge 40,Eclipse Black,True,256/8,256/8,800,"1,299"
4,133,Micromax 3,SAMSUNG Galaxy F14 5G,OMG Black,True,128/4,128/6,1020,"2,615"
5,152,micromax 3,Infinix HOT 20 5G,Racing Black,True,64/4,64/4,800,"2,490"
6,155,micromax 2,MOTOROLA Edge 40,Nebula Green,True,256/8,256/8,800,"1,299"
7,156,micromax 3,micromax 2,Nebula Green,True,256/4,256/8,1020,"2,499"
8,181,micromax,realme C31,Light Silver,True,32/3,32/3,2550,"1,375"
9,182,micromax1,micromax,Light Silver,True,32/3,32/3,800,"1,599"


**Interpretation — a scraping artefact, with strong evidence.**
In **all 16 rows the colour is identical to the previous row** (where several feature phones are consecutive, the value
is passed down the chain)
(e.g. row 118 *micromax 1* has colour *Eclipse Black*, 256 GB / 8 GB — exactly the Motorola Edge 40 above it).
Memory/RAM are usually copied too. A ₹1,299 phone with an 800 mAh battery and no camera cannot have 256 GB storage and
8 GB RAM; feature phones measure memory in **MB**, and the scraper appears to have filled the gap with the previous
listing's values.

**What we conclude:** `Colour`, `Memory` and `RAM` are **unreliable for these ~16 rows**. Battery, camera, display and
price look internally consistent for a feature phone.
**What we do *not* conclude yet:** whether to drop these rows, keep them with corrected/flagged values, or scope the
model to smartphones only. That is a Day 3 decision, and it affects the business scope (does the retailer price
feature phones with this model?).

### 11b. `Mobile Height` = 41.94

In [16]:
df[df["Model"].str.contains("G62", case=False)][["Unnamed: 0", "Model", "Colour", "Memory", "RAM", "Mobile Height", "Prize"]]

,Unnamed: 0,Model,Colour,Memory,RAM,Mobile Height,Prize
52,52,MOTOROLA G62 5G,Frosted Blue,128,6,16.64,"15,499"
57,57,MOTOROLA G62 5G,Midnight Gray,128,6,16.64,"15,499"
75,75,MOTOROLA G62 5G,Midnight Gray,128,8,41.94,"16,499"
88,88,MOTOROLA G62 5G,Frosted Blue,128,8,41.94,"16,499"


**Interpretation.** The same model (MOTOROLA G62 5G) has 16.64 in two rows and **41.94** in two others. No phone has a
42 cm display or body, and the model's own other listings say 16.64. This is a **data-entry error**, not an unusual
specification. Day 3 options: replace with the value from the same model's other rows (16.64) — justified by
within-dataset evidence — and document it.

### 11c. RAM looks capped at 8 GB, with “cheaper RAM, higher price” contradictions
Within one model and storage size, more RAM should never cost *less*. Let's find cases where it does.

In [17]:
check = df.assign(price=price_tmp, model_norm=model_norm)
contradictions = []
for (m, mem), g in check.groupby(["model_norm", "Memory"]):
    for _, lo in g.iterrows():
        for _, hi in g.iterrows():
            if lo["RAM"] < hi["RAM"] and lo["price"] > hi["price"]:
                contradictions.append(lo["Unnamed: 0"])
suspect_ram = check[check["Unnamed: 0"].isin(set(contradictions))]
print("Rows where a lower-RAM variant costs more than a higher-RAM variant of the same model & storage:",
      len(suspect_ram))
display(suspect_ram[["Unnamed: 0", "Model", "Colour", "Memory", "RAM", "Prize"]].sort_values(["Model", "RAM"]))
print("RAM values present in the data:", sorted(int(v) for v in df["RAM"].unique()))
print("\n256 GB phones recorded with RAM <= 4 GB (smartphones only):")
display(df[(df["Memory"] == 256) & (df["RAM"] <= 4) & (df["Battery_"] > 2600)][["Unnamed: 0", "Model", "RAM", "Prize"]])

Rows where a lower-RAM variant costs more than a higher-RAM variant of the same model & storage: 10


,Unnamed: 0,Model,Colour,Memory,RAM,Prize
53,53,POCO C50,Royal Blue,32,2,"6,999"
408,437,POCO F5 5G,Electric Blue,256,4,"32,999"
326,331,POCO F5 5G,Snowstorm White,256,6,"32,999"
461,523,POCO M4 Pro,Power Black,128,6,"16,499"
287,290,POCO M4 Pro 5G,Cool Blue,128,6,"16,999"
283,286,REDMI Note 12 Pro+ 5G,Arctic White,256,2,"32,999"
272,275,REDMI Note 12 Pro+ 5G,Iceberg Blue,256,4,"32,999"
358,371,REDMI Note 12 Pro+ 5G,Obsidian Black,256,4,"32,999"
414,446,Tecno Spark 9,Sky Mirror,64,3,"8,990"
243,243,redmi max,M3),16,2,"4,999"


RAM values present in the data: [2, 3, 4, 6, 8]

256 GB phones recorded with RAM <= 4 GB (smartphones only):


,Unnamed: 0,Model,RAM,Prize
272,275,REDMI Note 12 Pro+ 5G,4,"32,999"
283,286,REDMI Note 12 Pro+ 5G,2,"32,999"
358,371,REDMI Note 12 Pro+ 5G,4,"32,999"
364,379,vivo V23 5G,4,"27,540"
396,421,vivo V27 5G,4,"36,999"
400,427,MOTOROLA Edge 30 Ultra,4,"49,999"
408,437,POCO F5 5G,4,"32,999"
450,504,realme X3 SuperZoom,3,"17,999"
464,530,realme GT 2,2,"32,299"
522,616,IQOO Neo 7 5G,4,"34,688"


In [18]:
df[df["Model"].str.startswith("APPLE")][["Model", "Colour", "Memory", "RAM", "Processor_", "Prize"]]

,Model,Colour,Memory,RAM,Processor_,Prize
474,APPLE iPhone 11,Black,64,8,ios,"41,199"
493,APPLE iPhone 14 Plus,Midnight,128,6,IOS,"80,999"
520,APPLE iPhone 14 Plus,Purple,128,3,6 Core,"80,999"
525,APPLE iPhone 14 Plus,Starlight,128,4,ios,"80,999"
532,APPLE iPhone 12,Blue,64,8,ios,"53,999"


**Interpretation.**
- The RAM column only ever takes **2, 3, 4, 6, 8** — there is no 12 GB. The 10 flagged rows fall into two groups:
  - **Small price gaps** (POCO C50, Tecno Spark 9, POCO M4 Pro/M4 Pro 5G): a few hundred to a couple of thousand rupees,
    plausibly seller/date price noise (see §9) rather than wrong RAM. (*redmi max* is one of the feature-phone rows from §11a.)
  - **Premium 256 GB variants** (REDMI Note 12 Pro+, POCO F5): the most expensive variant records *less* RAM than the
    cheaper one (Note 12 Pro+ 256 GB: 8 GB at ₹29,999 but **2 GB** and **4 GB** at ₹32,999). The second table adds
    similar 256 GB rows that have no sibling to contradict (realme GT 2 at 2 GB, vivo V23, vivo V27, IQOO Neo 7, Edge 30 Ultra,
    realme X3 at 3–4 GB). This pattern is consistent with **“12 GB” being truncated** during scraping, though the
    dataset alone cannot prove the true value.
- **iPhone RAM is inconsistent:** three iPhone 14 Plus rows at the same price record RAM as 6, 3 and 4.
- **Decision deferred to Day 3**, with these options: (a) flag RAM as suspect for these rows and keep them,
  (b) correct only where the dataset itself gives evidence, (c) exclude. We will **not** silently overwrite values with
  numbers from outside sources without documenting them.
- **Consequence for interpretation:** any RAM–price relationship we find will be *understated* at the premium end.

### 11d. A mislabelled model

In [19]:
df[df["Model"].str.contains("Pixel 6a|nothing phone", case=False)][
    ["Unnamed: 0", "Model", "Colour", "RAM", "Battery_", "Rear Camera", "Front Camera", "Processor_", "Prize"]]

,Unnamed: 0,Model,Colour,RAM,Battery_,Rear Camera,Front Camera,Processor_,Prize
135,135,Google Pixel 6a,Chalk,6,4410,12MP,8MP,Google Tensor,"28,999"
136,136,Google Pixel 6a,Charcoal,6,4410,12MP,8MP,Google Tensor,"28,999"
328,333,nothing phone 1,White,8,4500,50MP,16MP,Qualcomm Snapdragon 778G+,"30,499"
329,334,Google Pixel 6a,chalk,8,4500,50MP,16MP,Qualcomm Snapdragon 778G+,"30,999"
342,349,nothing phone 1,Black,8,4500,50MP,16MP,Qualcomm Snapdragon 778G+,"29,499"


**Interpretation.** Two rows are named *Google Pixel 6a* with Google Tensor, 12 MP / 8 MP, 4410 mAh. A third row
(index 334, colour *chalk*) has **Snapdragon 778G+, 50 MP / 16 MP, 4500 mAh** — identical to the *nothing phone 1*
rows next to it. Its specs belong to a different phone. It is an inconsistent record; Day 3 should flag it (its spec
values are self-consistent, so the main risk is the **model label**, which matters for grouped splitting).

### 11e. Text inconsistencies: case, whitespace, encoding

In [20]:
def variants(series):
    key = series.str.strip().str.lower().str.replace(r"\s+", " ", regex=True)
    grouped = series.groupby(key).unique()
    return grouped[grouped.str.len() > 1]

print("MODEL names differing only by case/spacing:")
display(variants(df["Model"]).to_frame("raw variants"))

print("COLOUR names differing only by case/spacing:")
display(variants(df["Colour"]).to_frame("raw variants"))

print("Processor_ values with trailing whitespace:", int(df["Processor_"].str.endswith(" ").sum()))
print("Processor_ distinct raw:", df["Processor_"].nunique(),
      "| after strip+lower:", df["Processor_"].str.strip().str.lower().nunique())

non_ascii = df[df.apply(lambda r: any(not str(v).isascii() for v in r), axis=1)]
print("\nRows with non-ASCII (mis-encoded) characters:")
non_ascii[["Unnamed: 0", "Model", "Colour", "Processor_"]]

MODEL names differing only by case/spacing:


,raw variants
Model,
infinix hot 20 5g,"[Infinix HOT 20 5G, Infinix Hot 20 5G]"
micromax 3,"[Micromax 3, micromax 3]"
motorola g42,"[MOTOROLA g42, MOTOROLA G42]"
oppo a17k,"[OPPO A17k, OPPO A17K]"
realme narzo 50a prime,"[realme Narzo 50A Prime, realme NARZO 50A PRIME]"
realme narzo 50i prime,"[realme Narzo 50i Prime, realme narzo 50i Prime]"


COLOUR names differing only by case/spacing:


,raw variants
Colour,
chalk,"[Chalk, chalk]"
flash blue,"[Flash Blue, FLASH BLUE]"
matte black,"[Matte Black, Matte black]"
uyuni blue,"[Uyuni Blue, UYUNI BLUE]"


Processor_ values with trailing whitespace: 18
Processor_ distinct raw: 123 | after strip+lower: 113

Rows with non-ASCII (mis-encoded) characters:


,Unnamed: 0,Model,Colour,Processor_
270,273,Infinix Hot 12,7Â° Purple,Meditek Helio G37
458,520,OnePlus Nord,Blue Marble,QualcommÂ® Snapdragonâ_x0084_¢ 765G


In [21]:
# Same chipset written many ways — sample of Snapdragon 680 / Helio G37 spellings
for pattern in ["680", "G37"]:
    print(pattern, "->", sorted(df.loc[df["Processor_"].str.contains(pattern), "Processor_"].unique()))

680 -> ['Qualcomm SM6225 Snapdragon 680 4G (6 nm)', 'Qualcomm Snapdragon 680', 'Qualcomm Snapdragon 680 (SM6225)', 'Snapdragon 680']
G37 -> ['G37', 'MediaTek G37', 'MediaTek Helio G37', 'Mediatek Helio G37', 'Meditek Helio G37']


**Interpretation.**
- **6 model names** and **4 colours** differ only by capitalisation (e.g. *OPPO A17k* vs *OPPO A17K*). Untreated, the
  model would see them as different phones and a grouped split would put “the same” phone in both train and test.
- **Processor_** is the messiest column: 123 raw values collapse to 113 after basic normalisation, and one chipset
  appears under many spellings (*Snapdragon 680* is written five ways; *Helio G37* includes the typo *Meditek*).
- **2 rows contain mis-encoded characters** (UTF-8 read as Latin-1): `7Â° Purple` and
  `QualcommÂ® Snapdragonâ_x0084_¢ 765G` (should be *Qualcomm® Snapdragon™ 765G*).
- Model names include some odd entries: `micromax1`, `I Kall Z19Pro Flash blue` (colour inside the model name),
  `vivo Y565G` (= *vivo Y56 5G*), and `redmi min / max / mono` with colour `M3)` — these last look like feature phones
  whose names were parsed incorrectly.

### 11f. Other observations

In [22]:
print("Index column: min", df["Unnamed: 0"].min(), "| max", df["Unnamed: 0"].max(),
      "| rows", len(df), "| missing index numbers", df["Unnamed: 0"].max() + 1 - len(df),
      "| strictly increasing:", df["Unnamed: 0"].is_monotonic_increasing)

ai_brand = df.loc[df["AI Lens"] == 1, "Model"].str.split().str[0].value_counts()
print("\nBrands with AI Lens = 1:", ai_brand.to_dict())
print("Infinix rows with AI Lens = 0:",
      int(((df["Model"].str.startswith("Infinix")) & (df["AI Lens"] == 0)).sum()),
      "of", int(df["Model"].str.startswith("Infinix").sum()))

print("\nBattery outside 3000–7000 mAh:", int((~df["Battery_"].between(3000, 7000)).sum()), "rows")
print("Max battery row:")
print(df.loc[df["Battery_"].idxmax(), ["Model", "Battery_", "Prize"]].to_dict())

Index column: min 0 | max 642 | rows 541 | missing index numbers 102 | strictly increasing: True

Brands with AI Lens = 1: {'Infinix': 34}
Infinix rows with AI Lens = 0: 31 of 65

Battery outside 3000–7000 mAh: 17 rows
Max battery row:
{'Model': 'Tecno Pova 3', 'Battery_': 7000, 'Prize': '13,999'}


**Interpretation.**
- **`Unnamed: 0`** runs 0→642 but there are only 541 rows → **102 index numbers are missing**. Rows were removed before
  the file was shared (the name *Processed_* fits this). It is an artefact, not a feature, and will be dropped on Day 3.
- **AI Lens is set only for Infinix phones** (34 rows), and not even for all of them. It probably reflects how one
  brand's listings describe their camera rather than a comparable hardware feature across brands. Any importance it
  shows later may really be a **brand effect** — keep this in mind on Days 5–6.
- The 7000 mAh battery (Tecno Pova 3) is a real phone with a genuinely large battery — **valid extreme**, keep.

## 12. Data dictionary

In [23]:
data_dictionary = pd.DataFrame([
    ["Unnamed: 0", "int", "Leftover row index from an earlier export (0–642, 102 gaps)", "—", "Identifier only", "Drop"],
    ["Model", "text", "Brand + model name of the phone", "—", "187 raw / ~181 normalised values; case variants; some feature-phone names garbled", "Extract brand; use as grouping key for CV"],
    ["Colour", "text", "Marketing colour name of the listing", "—", "275 values; 'M3)' fragments; copied from previous row on feature phones", "Likely exclude from features"],
    ["Memory", "int", "Internal storage", "GB", "Values 16/32/64/128/256; unreliable on ~16 feature-phone rows", "Numeric (ordinal)"],
    ["RAM", "int", "Memory (RAM)", "GB", "Values 2/3/4/6/8 only; suspected truncation of 12 GB; inconsistent for iPhones and feature phones", "Numeric (ordinal), flag suspect rows"],
    ["Battery_", "int", "Battery capacity", "mAh", "800–7000; <2600 only on feature phones", "Numeric"],
    ["Rear Camera", "text", "Primary rear camera resolution", "MP", "'0MP' = no camera (10 rows)", "Parse to number"],
    ["Front Camera", "text", "Front camera resolution", "MP", "'0MP' = no camera (16 rows)", "Parse to number"],
    ["AI Lens", "int (0/1)", "Listing mentions an AI lens", "flag", "Only Infinix rows are 1 → possible brand proxy", "Binary"],
    ["Mobile Height", "float", "Display size (diagonal) as listed on Flipkart — despite the column name", "cm", "41.94 on two MOTOROLA G62 rows = error", "Numeric; fix error"],
    ["Processor_", "text", "Chipset description", "—", "123 values, many spellings of same chip; ~25 rows contain OS/core count/vendor only; encoding error", "Engineer vendor / family / tier"],
    ["Prize", "text", "Selling price (target)", "INR (₹)", "Comma-formatted string; all parse cleanly; right-skewed", "Convert to int; rename to price"],
], columns=["column", "raw type", "meaning", "unit", "quality notes", "planned treatment"])
data_dictionary

,column,raw type,meaning,unit,quality notes,planned treatment
0,Unnamed: 0,int,Leftover row index from an earlier export (0–6...,—,Identifier only,Drop
1,Model,text,Brand + model name of the phone,—,187 raw / ~181 normalised values; case variant...,Extract brand; use as grouping key for CV
2,Colour,text,Marketing colour name of the listing,—,275 values; 'M3)' fragments; copied from previ...,Likely exclude from features
3,Memory,int,Internal storage,GB,Values 16/32/64/128/256; unreliable on ~16 fea...,Numeric (ordinal)
4,RAM,int,Memory (RAM),GB,Values 2/3/4/6/8 only; suspected truncation of...,"Numeric (ordinal), flag suspect rows"
5,Battery_,int,Battery capacity,mAh,800–7000; <2600 only on feature phones,Numeric
6,Rear Camera,text,Primary rear camera resolution,MP,'0MP' = no camera (10 rows),Parse to number
7,Front Camera,text,Front camera resolution,MP,'0MP' = no camera (16 rows),Parse to number
8,AI Lens,int (0/1),Listing mentions an AI lens,flag,Only Infinix rows are 1 → possible brand proxy,Binary
9,Mobile Height,float,Display size (diagonal) as listed on Flipkart ...,cm,41.94 on two MOTOROLA G62 rows = error,Numeric; fix error


The same dictionary is maintained as Markdown in `reports/data_dictionary.md` for the README and the final report.

## 13. Day 1 summary

| Area | Finding | Severity | Decided on |
|---|---|---|---|
| Structure | 541 × 12, schema as described | — | ✅ verified |
| Target | `Prize` is a comma string; all 541 parse; ₹920–₹80,999; skew ≈ 2.43 | Medium | Day 3 (convert), Day 7 (log target?) |
| Missing | No NaNs; disguised gaps in `Processor_` (~25 rows) and `Colour` ("M3)") | Medium | Day 3–4 |
| Duplicates | 10 duplicate listings ignoring the index | Medium | Day 3 |
| Near-duplicates | ~181 models / ~280 configs across 541 rows | **High (validation)** | Day 7 – grouped CV |
| Feature phones | ~16 rows; colour/memory/RAM copied from previous row | **High** | Day 3 – scope decision |
| RAM | Max 8; contradictions suggest truncated 12 GB on ~10 premium rows | **High** | Day 3 |
| Mobile Height | Actually display size (cm); 41.94 is an error (2 rows) | Medium | Day 3 |
| Mislabel | One “Pixel 6a” row has Nothing Phone 1 specs | Low | Day 3 |
| Text hygiene | Case variants, trailing spaces, mis-encoding, many chipset spellings | Medium | Day 3–4 |
| Outliers | 25 premium phones above IQR fence — **genuine** | — | Keep |
| AI Lens | Infinix-only → brand proxy risk | Interpretation | Days 5–6 |

**Nothing has been modified.** `data/raw/Processed_Flipdata.csv` is unchanged; cleaning starts on Day 3 after EDA.